# 05 — End-to-End Redaction Pipeline

```
input_folder/*.pdf
  → temp_images/{stem}_page_{n}.png          (PyMuPDF render)
  → redacted_text/{stem}_page_{n}.json       (Bedrock: sanitized text + mapping)
  → output_folder/redacted_{stem}.pdf        (ReportLab: content + summary table)
  → cleanup temp_images/ + redacted_text/
```

PII/PHI is **replaced with realistic fictitious dummy values** — consistent across all pages.
A **Summary Table** listing every replacement is appended as the final page of each output PDF.

Edit the **Configuration** cell, then **Run All**.

In [ ]:
%pip install pymupdf boto3 Pillow reportlab --prefer-binary -q

In [ ]:
# ── Configuration ──────────────────────────────────────────────
import os
from pathlib import Path

BASE_DIR      = Path(os.getcwd()).parent
INPUT_FOLDER  = BASE_DIR / "input_folder"
OUTPUT_FOLDER = BASE_DIR / "output_folder"
TEMP_FOLDER   = BASE_DIR / "temp_images"
TEXT_FOLDER   = BASE_DIR / "redacted_text"

AWS_REGION    = "us-east-2"
BEDROCK_MODEL = "us.anthropic.claude-3-7-sonnet-20250219-v1:0"
PAGE_DPI      = 150
MAX_TOKENS    = 4096
RETRY_LIMIT   = 3
RETRY_DELAY   = 5
CLEAN_UP      = True

for folder in [OUTPUT_FOLDER, TEMP_FOLDER, TEXT_FOLDER]:
    folder.mkdir(exist_ok=True)

print("Configuration OK")

In [ ]:
# ── Imports & logger ───────────────────────────────────────────
import fitz, boto3, json, base64, time
from collections import defaultdict

from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import cm
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, PageBreak, Table, TableStyle
)
from reportlab.lib.enums import TA_LEFT, TA_CENTER

from utils import get_logger, extract_json
logger = get_logger("05_pipeline")

bedrock = boto3.client("bedrock-runtime", region_name=AWS_REGION)
logger.info("Imports OK — model: %s", BEDROCK_MODEL)

In [ ]:
# ── Prompt + helper functions ──────────────────────────────────

_BASE_PROMPT = """You are a data-sanitization assistant. Your task is to take the provided document \
and produce a new, clean version in which all PII (Personally Identifiable Information) and \
PHI (Protected Health Information) are fully redacted and replaced with realistic but \
completely fictitious dummy records. Use the following definitions of PII/PHI when identifying \
sensitive information:
- Full names in ANY format or order: "First Last", "Last, First", "Last, First Middle", titles \
(Mr., Dr., etc.), suffixes (Jr., Sr., III), and initials. This includes names preceded by role \
labels such as "Claimant:", "Patient:", "Provider:", "Insured:", "Applicant:", "Member:", \
"Beneficiary:", etc.
- Email addresses
- Phone numbers
- Social Security Numbers (SSN) or national identifiers
- Physical mailing addresses
- Dates of Birth (DOB)
- Medical record numbers or identifiers
- Any specific medical diagnoses or conditions tied to individuals

Requirements:
1. Identify every occurrence of PII/PHI within the document.
2. Replace each sensitive item with a consistent, fictitious dummy value.
   If the same value appears multiple times, use the SAME replacement throughout.
3. Dummy values must follow valid formats but must NOT correspond to real individuals.
4. Maintain the original meaning, readability, and structure — only sensitive data is substituted.
5. Pay special attention to names in inverted "Last, First" format, names inside table cells or \
form fields, and names preceded by role/label prefixes. These are all PII regardless of \
formatting or context.

<<PRIOR_MAPPING>>
Return your response as valid JSON with exactly this structure (no markdown fences, no extra keys):
{
  "sanitized_text": "<full sanitized page text, preserving line breaks as \\n>",
  "mapping": [
    {"original_masked": "J*** S***", "replacement": "Alex Carter", "type": "Name"},
    ...
  ]
}

Now process the following document:"""


def build_prompt(prior_mapping: list[dict]) -> str:
    if not prior_mapping:
        return _BASE_PROMPT.replace("<<PRIOR_MAPPING>>\n", "")
    rows = "\n".join(
        f"  - {r['original_masked']} → {r['replacement']} ({r['type']})"
        for r in prior_mapping
    )
    section = (
        "Previously identified mappings from earlier pages — "
        "use these EXACT replacements for any matching values on this page:\n"
        + rows + "\n\n"
    )
    return _BASE_PROMPT.replace("<<PRIOR_MAPPING>>", section.rstrip())


def pdf_to_images(pdf_path: Path) -> list[Path]:
    logger.info("Rendering pages: %s", pdf_path.name)
    doc  = fitz.open(str(pdf_path))
    zoom = PAGE_DPI / 72
    mat  = fitz.Matrix(zoom, zoom)
    paths = []
    for i in range(len(doc)):
        pix  = doc.load_page(i).get_pixmap(matrix=mat, alpha=False)
        dest = TEMP_FOLDER / f"{pdf_path.stem}_page_{i+1}.png"
        pix.save(str(dest))
        paths.append(dest)
        logger.debug("  Page %d → %s (%dx%d)", i+1, dest.name, pix.width, pix.height)
    doc.close()
    logger.info("%d page(s) rendered for %s", len(paths), pdf_path.name)
    return paths


def redact_image(image_path: Path, prior_mapping: list[dict]) -> dict:
    with open(image_path, "rb") as f:
        img_b64 = base64.standard_b64encode(f.read()).decode()

    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": MAX_TOKENS,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image",
                 "source": {"type": "base64", "media_type": "image/png", "data": img_b64}},
                {"type": "text", "text": build_prompt(prior_mapping)}
            ]
        }]
    }

    last_exc = None
    for attempt in range(1, RETRY_LIMIT + 1):
        try:
            logger.debug("Bedrock call: %s (attempt %d)", image_path.name, attempt)
            resp = bedrock.invoke_model(
                modelId=BEDROCK_MODEL, body=json.dumps(body),
                contentType="application/json", accept="application/json"
            )
            raw = json.loads(resp["body"].read())["content"][0]["text"]
            logger.debug("Raw response (first 200 chars): %s", raw[:200].replace("\n", "\\n"))
            return extract_json(raw)

        except json.JSONDecodeError as exc:
            logger.warning(
                "Page %s: non-JSON response (image-only/banner page). "
                "Using raw text, empty mapping. Preview: %s",
                image_path.name, exc.doc[:200].replace("\n", "\\n")
            )
            return {"sanitized_text": exc.doc.strip(), "mapping": []}

        except Exception as exc:
            last_exc = exc
            logger.warning("Attempt %d/%d failed for %s: %s",
                           attempt, RETRY_LIMIT, image_path.name, exc)
            if attempt < RETRY_LIMIT:
                time.sleep(RETRY_DELAY)

    logger.error("All retries exhausted for %s", image_path.name)
    raise last_exc


def merge_mapping(accumulated: list[dict], new_rows: list[dict]) -> list[dict]:
    seen = {(r["original_masked"], r["type"]) for r in accumulated}
    added = 0
    for row in new_rows:
        key = (row["original_masked"], row["type"])
        if key not in seen:
            accumulated.append(row)
            seen.add(key)
            added += 1
    if added:
        logger.debug("Mapping +%d entries (total %d)", added, len(accumulated))
    return accumulated


def xml_escape(text: str) -> str:
    return text.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")


def build_redacted_pdf(pdf_stem: str, page_texts: list[str], out_path: Path) -> None:
    """Write sanitized content pages only — no summary table."""
    logger.info("Building redacted PDF: %s (%d pages)", out_path.name, len(page_texts))
    doc = SimpleDocTemplate(
        str(out_path), pagesize=A4,
        leftMargin=2*cm, rightMargin=2*cm,
        topMargin=2*cm, bottomMargin=2*cm,
    )
    styles = getSampleStyleSheet()
    body_style = ParagraphStyle(
        "Body", parent=styles["Normal"],
        fontName="Courier", fontSize=9, leading=13, alignment=TA_LEFT,
    )
    story = []
    for idx, text in enumerate(page_texts):
        if idx > 0:
            story.append(PageBreak())
        logger.debug("  Page %d: %d chars", idx + 1, len(text))
        for line in text.splitlines():
            safe = xml_escape(line)
            story.append(Paragraph(safe, body_style) if safe.strip() else Spacer(1, 6))
    doc.build(story)
    logger.info("Written: %s (%.1f KB)", out_path.name, out_path.stat().st_size / 1024)


def build_summary_pdf(pdf_stem: str, mapping: list[dict], out_path: Path) -> None:
    """Write a standalone summary PDF with the PII/PHI replacement table."""
    logger.info("Building summary PDF: %s (%d entries)", out_path.name, len(mapping))
    doc = SimpleDocTemplate(
        str(out_path), pagesize=A4,
        leftMargin=2*cm, rightMargin=2*cm,
        topMargin=2*cm, bottomMargin=2*cm,
    )
    styles = getSampleStyleSheet()
    heading_style = ParagraphStyle(
        "Heading", parent=styles["Heading2"], alignment=TA_CENTER, spaceAfter=12,
    )
    story = [
        Paragraph(f"PII / PHI Replacement Summary — {pdf_stem}", heading_style),
        Spacer(1, 6),
    ]
    table_data = [["Original (masked)", "Replacement", "Data Type"]]
    for row in mapping:
        table_data.append([row.get("original_masked", ""),
                           row.get("replacement", ""),
                           row.get("type", "")])
    tbl = Table(table_data, colWidths=[6*cm, 7*cm, 4*cm], repeatRows=1)
    tbl.setStyle(TableStyle([
        ("BACKGROUND",    (0, 0), (-1, 0),  colors.HexColor("#2E4057")),
        ("TEXTCOLOR",     (0, 0), (-1, 0),  colors.white),
        ("FONTNAME",      (0, 0), (-1, 0),  "Helvetica-Bold"),
        ("FONTSIZE",      (0, 0), (-1, -1), 8),
        ("ROWBACKGROUNDS",(0, 1), (-1, -1), [colors.white, colors.HexColor("#F2F2F2")]),
        ("GRID",          (0, 0), (-1, -1), 0.4, colors.grey),
        ("VALIGN",        (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING",   (0, 0), (-1, -1), 4),
        ("RIGHTPADDING",  (0, 0), (-1, -1), 4),
        ("TOPPADDING",    (0, 0), (-1, -1), 3),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
    ]))
    story.append(tbl)
    doc.build(story)
    logger.info("Written: %s (%.1f KB)", out_path.name, out_path.stat().st_size / 1024)


logger.info("Prompt + helper functions defined")

In [ ]:
# ── Main pipeline ──────────────────────────────────────────────
pdf_files = sorted(
    p for p in INPUT_FOLDER.iterdir()
    if p.is_file() and p.suffix.lower() == ".pdf"
)
logger.info("Found %d PDF(s) in input_folder/", len(pdf_files))

if not pdf_files:
    raise FileNotFoundError("No PDFs found. Add files to input_folder/ and re-run.")

for pdf_path in pdf_files:
    redacted_path = OUTPUT_FOLDER / f"redacted_{pdf_path.stem}.pdf"
    summary_path  = OUTPUT_FOLDER / f"summary_{pdf_path.stem}.pdf"

    if redacted_path.exists() and summary_path.exists():
        logger.info("SKIP (outputs exist): %s", pdf_path.name)
        continue

    logger.info("=" * 55)
    logger.info("START: %s", pdf_path.name)

    # Step 1 — PDF → images
    logger.info("[1/3] PDF → images")
    image_paths = pdf_to_images(pdf_path)

    # Step 2 — Redact via Bedrock
    logger.info("[2/3] Redacting via Bedrock")
    page_texts: list[str] = []
    accumulated_mapping: list[dict] = []

    for img_path in image_paths:
        cache = TEXT_FOLDER / f"{img_path.stem}.json"
        if cache.exists():
            logger.info("  [cache] %s", img_path.name)
            data = json.loads(cache.read_text(encoding="utf-8"))
        else:
            logger.info("  Redacting %s ...", img_path.name)
            data = redact_image(img_path, prior_mapping=accumulated_mapping)
            cache.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
            logger.info("  Done: %d entries this page", len(data.get("mapping", [])))

        page_texts.append(data.get("sanitized_text", ""))
        accumulated_mapping = merge_mapping(accumulated_mapping, data.get("mapping", []))

    logger.info("%d unique replacements across all pages", len(accumulated_mapping))

    # Step 3 — Build output PDFs
    logger.info("[3/3] Building output PDFs")
    build_redacted_pdf(pdf_path.stem, page_texts, redacted_path)
    build_summary_pdf(pdf_path.stem, accumulated_mapping, summary_path)

    # Cleanup
    if CLEAN_UP:
        for img_path in image_paths:
            img_path.unlink(missing_ok=True)
        for jf in TEXT_FOLDER.glob(f"{pdf_path.stem}_page_*.json"):
            jf.unlink(missing_ok=True)
        logger.info("Temp files cleaned")

    logger.info("DONE: %s", pdf_path.name)

logger.info("=" * 55)
logger.info("Pipeline complete — %d PDF(s) processed", len(pdf_files))

In [ ]:
# ── Final summary ──────────────────────────────────────────────
for p in sorted(OUTPUT_FOLDER.glob("redacted_*.pdf")):
    logger.info("  %s  (%.1f KB)", p.name, p.stat().st_size / 1024)